# Cell 1 — Load lesson pages

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

print(f"Files downloaded: {len(files)}")

Files downloaded: 72


# Cell 2 — Convert to documents

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print(f"Documents loaded: {len(documents)}")

Documents loaded: 72


# Cell 3 — Inspect structure

In [3]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

# Cell 4 — Build search index

In [4]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

# Cell 5 — Question 2 search

In [5]:
query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(
    query=query,
    num_results=5
)

results[:3]

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

# Cell 6 — Get first result

In [6]:
results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

# First inspect a document

In [7]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [8]:
from rag_helper import RAGBase

In [9]:
class LessonRAG(RAGBase):

    def search(self, query, num_results=5):

        return self.index.search(
            query=query,
            num_results=num_results
        )

    def build_context(self, search_results):

        lines = []

        for doc in search_results:
            lines.append(f"FILE: {doc['filename']}")
            lines.append(doc['content'])
            lines.append("")

        return "\n".join(lines)

    def llm(self, prompt):

        input_messages = [
            {
                "role": "developer",
                "content": self.instructions
            },
            {
                "role": "user",
                "content": prompt
            }
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response

    def rag(self, query):

        search_results = self.search(query)

        prompt = self.build_prompt(
            query,
            search_results
        )

        response = self.llm(prompt)

        return {
            "answer": response.output_text,
            "usage": response.usage,
            "response": response
        }

In [11]:
from dotenv import load_dotenv

load_dotenv()

True

In [12]:
import os

print(os.getenv("OPENAI_API_KEY")[:15] + "...")

sk-proj-VbqhR9L...


# Create OpenAI client

In [13]:
from openai import OpenAI

client = OpenAI()

# Create RAG instance

In [14]:
lesson_rag = LessonRAG(
    index=index,
    llm_client=client
)

# Run the homework query

In [15]:
result = lesson_rag.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

In [16]:
result

{'answer': 'The loop keeps calling the model inside a `while True` loop. After each model response, it checks whether any `function_call` items were returned:\n\n- If there are function calls, the code runs them, appends the tool outputs to `messages`, and loops again.\n- If there are no function calls, it breaks out of the loop and stops.\n\nSo the stop condition is simply: **no function calls in the latest response**.',
 'usage': ResponseUsage(input_tokens=7126, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=94, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7220),
 'response': Response(id='resp_00865efe077da966006a259e1ea3a4819db2d7527080cd9ced', created_at=1780850206.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_00865efe077da966006a259e1f4bf8819d8b5744bae2622231', content=[ResponseOutputText(annotations=[], text='T

In [17]:
print(result["answer"])

The loop keeps calling the model inside a `while True` loop. After each model response, it checks whether any `function_call` items were returned:

- If there are function calls, the code runs them, appends the tool outputs to `messages`, and loops again.
- If there are no function calls, it breaks out of the loop and stops.

So the stop condition is simply: **no function calls in the latest response**.


In [18]:
print(result["answer"])

The loop keeps calling the model inside a `while True` loop. After each model response, it checks whether any `function_call` items were returned:

- If there are function calls, the code runs them, appends the tool outputs to `messages`, and loops again.
- If there are no function calls, it breaks out of the loop and stops.

So the stop condition is simply: **no function calls in the latest response**.


In [19]:
print(result["usage"].input_tokens)

7126


In [21]:
print(result["usage"])

ResponseUsage(input_tokens=7126, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=94, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7220)


In [22]:
result["usage"]

ResponseUsage(input_tokens=7126, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=94, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7220)

In [24]:
dir(result["usage"])

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_extra_info__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_on_complete__',
 '__pydantic_parent_namespace__',
 '__pydantic_post_init__',
 '__pydantic_private__',
 '__pydantic_root_model__

# Q4 – Chunking

In [26]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

len(chunks)

295

# Q5 – RAG with chunking

# Build a chunk index

In [27]:
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

# Create a chunked RAG

In [28]:
chunk_rag = LessonRAG(
    index=chunk_index,
    llm_client=client
)

# Run the same query

In [29]:
chunk_result = chunk_rag.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

In [30]:
chunk_result["usage"].model_dump()

{'input_tokens': 2309,
 'input_tokens_details': {'cached_tokens': 0},
 'output_tokens': 99,
 'output_tokens_details': {'reasoning_tokens': 0},
 'total_tokens': 2408}

In [31]:
len(chunks)

295

# Q6 – Agent

The actual number depends on:

model
framework
tool usage

After Q4 and Q5 are complete we'll implement the search tool and run the agent once. Then we'll count the actual search calls and select the closest option.

In [32]:
chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

len(chunks)

295

# Q6: Turning it into an Agent

# Cell 1 — Global counter

In [33]:
search_calls = 0

# Cell 2 — Search tool

In [34]:
def search_lessons(query: str) -> str:
    """
    Search the LLM Zoomcamp lesson chunks.
    Returns the most relevant lesson content.
    """

    global search_calls
    search_calls += 1

    results = chunk_index.search(
        query=query,
        num_results=3
    )

    docs = []

    for doc in results:
        docs.append(
            f"FILE: {doc['filename']}\n{doc['content']}"
        )

    return "\n\n".join(docs)

# Cell 3 — Import ToyAIKit

In [36]:

import toyaikit

print(toyaikit.__file__)
print(dir(toyaikit))

C:\Users\dmish\llm-zoomcamp-code\.venv\Lib\site-packages\toyaikit\__init__.py
['AnthropicClient', 'AnthropicMessagesRunner', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'chat', 'llm', 'pricing', 'tools']


In [37]:
import toyaikit

print(toyaikit.__file__)
print(dir(toyaikit))

C:\Users\dmish\llm-zoomcamp-code\.venv\Lib\site-packages\toyaikit\__init__.py
['AnthropicClient', 'AnthropicMessagesRunner', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'chat', 'llm', 'pricing', 'tools']


In [38]:
import inspect
import toyaikit.chat

print(dir(toyaikit.chat))

['ChatAssistant', 'CostInfo', 'IPythonChatInterface', 'LoopResult', 'OpenAIClient', 'TokenUsage', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'chat', 'interface', 'runners']


In [39]:
import inspect
import toyaikit

print(inspect.getsource(toyaikit.chat))

from toyaikit.chat.chat import ChatAssistant
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import LoopResult
from toyaikit.llm import OpenAIClient
from toyaikit.pricing import CostInfo, TokenUsage

__all__ = [
    "ChatAssistant",
    "OpenAIClient",
    "IPythonChatInterface",
    "LoopResult",
    "TokenUsage",
    "CostInfo",
]



In [40]:
help(toyaikit.chat)


Help on package toyaikit.chat in toyaikit:

NAME
    toyaikit.chat

PACKAGE CONTENTS
    chat
    interface
    runners

CLASSES
    builtins.object
        toyaikit.chat.chat.ChatAssistant
        toyaikit.chat.interface.IPythonChatInterface
        toyaikit.pricing.CostInfo
        toyaikit.pricing.TokenUsage
    toyaikit.llm.LLMClient(builtins.object)
        toyaikit.llm.OpenAIClient
    typing.Generic(builtins.object)
        toyaikit.chat.runners.LoopResult

    class ChatAssistant(builtins.object)
     |  ChatAssistant(tools: toyaikit.tools.Tools, developer_prompt: str, chat_interface: toyaikit.chat.interface.ChatInterface, llm_client: toyaikit.llm.LLMClient)
     |
     |  Methods defined here:
     |
     |  __init__(self, tools: toyaikit.tools.Tools, developer_prompt: str, chat_interface: toyaikit.chat.interface.ChatInterface, llm_client: toyaikit.llm.LLMClient)
     |      Initialize self.  See help(type(self)) for accurate signature.
     |
     |  run(self)
     |
     |  

In [41]:
print(dir(toyaikit.chat))
print(dir(toyaikit.llm))
print(dir(toyaikit.tools))

['ChatAssistant', 'CostInfo', 'IPythonChatInterface', 'LoopResult', 'OpenAIClient', 'TokenUsage', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'chat', 'interface', 'runners']
['AnthropicClient', 'BaseModel', 'ChatCompletion', 'LLMClient', 'List', 'Message', 'OpenAI', 'OpenAIChatCompletionsClient', 'OpenAIClient', 'Optional', 'ParsedChatCompletion', 'ParsedResponse', 'RawMessageStopEvent', 'Response', 'Tools', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__']
['FunctionCallOutput', 'Tools', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'generate_function_schema', 'generate_schemas_from_instance', 'get_instance_methods', 'get_type_hints', 'inspect', 'json', 'python_type_to_json_type', 'wrap_instance_methods']


In [42]:
search_calls = 0

def search_lessons(query):
    global search_calls
    search_calls += 1

    results = chunk_index.search(
        query=query,
        num_results=3
    )

    return "\n\n".join(
        f"FILE: {r['filename']}\n{r['content']}"
        for r in results
    )

# Create the Tool Schema

In [43]:
tools = [
    {
        "type": "function",
        "name": "search_lessons",
        "description": "Search the LLM Zoomcamp lesson chunks",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string"
                }
            },
            "required": ["query"]
        }
    }
]

# Then run an agentic loop.

In [44]:
from toyaikit.chat import ChatAssistant
import inspect

print(inspect.signature(ChatAssistant))

(tools: toyaikit.tools.Tools, developer_prompt: str, chat_interface: toyaikit.chat.interface.ChatInterface, llm_client: toyaikit.llm.LLMClient)


In [45]:
from toyaikit.chat import IPythonChatInterface

help(IPythonChatInterface)

Help on class IPythonChatInterface in module toyaikit.chat.interface:

class IPythonChatInterface(builtins.object)
 |  Methods defined here:
 |
 |  display(self, message: str) -> None
 |
 |  display_function_call(self, function_name: str, arguments: str, result: str) -> None
 |
 |  display_reasoning(self, markdown_text: str) -> None
 |
 |  display_response(self, markdown_text: str) -> None
 |
 |  input(self) -> str
 |
 |  ----------------------------------------------------------------------
 |  Data descriptors defined here:
 |
 |  __dict__
 |      dictionary for instance variables
 |
 |  __weakref__
 |      list of weak references to the object



In [46]:
from toyaikit.llm import OpenAIClient

help(OpenAIClient)

Help on class OpenAIClient in module toyaikit.llm:

class OpenAIClient(LLMClient)
 |  OpenAIClient(model: str = 'gpt-4o-mini', client: openai.OpenAI = None, extra_kwargs: dict = None)
 |
 |  Method resolution order:
 |      OpenAIClient
 |      LLMClient
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, model: str = 'gpt-4o-mini', client: openai.OpenAI = None, extra_kwargs: dict = None)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  send_request(self, chat_messages: List, tools: toyaikit.tools.Tools = None, output_format: pydantic.main.BaseModel = None) -> openai.types.responses.response.Response | openai.types.responses.parsed_response.ParsedResponse
 |
 |  ----------------------------------------------------------------------
 |  Data descriptors inherited from LLMClient:
 |
 |  __dict__
 |      dictionary for instance variables
 |
 |  __weakref__
 |      list of weak references to the object



In [47]:
from toyaikit.tools import Tools

help(Tools)

Help on class Tools in module toyaikit.tools:

class Tools(builtins.object)
 |  Methods defined here:
 |
 |  __init__(self)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  add_tool(self, function, schema=None)
 |      Add a tool to the Tools object.
 |
 |      For the schema, this is the format:
 |
 |      {
 |          "type": "function",
 |          "name": "<function_name>",
 |          "description": "<Brief explanation of what the function does>",
 |          "parameters": {
 |              "type": "object",
 |              "properties": {
 |              "<parameter_name>": {
 |                  "type": "<data_type>",  // e.g., "string", "number", "boolean", "array", "object"
 |                  "description": "<Description of the parameter's purpose>"
 |              }
 |              // Additional parameters can be added here
 |              },
 |              "required": ["<param1>", "<param2>"],  // List all required parameters here
 |           

In [48]:
import inspect

from toyaikit.chat import IPythonChatInterface
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools

print(inspect.signature(IPythonChatInterface))
print(inspect.signature(OpenAIClient))
print(inspect.signature(Tools))

()
(model: str = 'gpt-4o-mini', client: openai.OpenAI = None, extra_kwargs: dict = None)
()


In [49]:
import inspect
from toyaikit.tools import Tools

print(dir(Tools))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'add_tool', 'add_tools', 'function_call', 'get_tools']


In [50]:
help(Tools)

Help on class Tools in module toyaikit.tools:

class Tools(builtins.object)
 |  Methods defined here:
 |
 |  __init__(self)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  add_tool(self, function, schema=None)
 |      Add a tool to the Tools object.
 |
 |      For the schema, this is the format:
 |
 |      {
 |          "type": "function",
 |          "name": "<function_name>",
 |          "description": "<Brief explanation of what the function does>",
 |          "parameters": {
 |              "type": "object",
 |              "properties": {
 |              "<parameter_name>": {
 |                  "type": "<data_type>",  // e.g., "string", "number", "boolean", "array", "object"
 |                  "description": "<Description of the parameter's purpose>"
 |              }
 |              // Additional parameters can be added here
 |              },
 |              "required": ["<param1>", "<param2>"],  // List all required parameters here
 |           

In [51]:
import inspect
from toyaikit.chat import ChatAssistant

print([m for m in dir(ChatAssistant) if not m.startswith("_")])

['run']


In [52]:
print(dir(Tools))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'add_tool', 'add_tools', 'function_call', 'get_tools']


In [53]:
print([m for m in dir(ChatAssistant) if not m.startswith("_")])

['run']


# Cell 1 — Create the search tool

In [54]:
search_calls = 0

def search_lessons(query: str) -> str:
    """
    Search the LLM Zoomcamp lesson chunks.
    """

    global search_calls
    search_calls += 1

    results = chunk_index.search(
        query=query,
        num_results=3
    )

    return "\n\n".join(
        f"FILE: {r['filename']}\n{r['content']}"
        for r in results
    )

# Cell 2 — Create Tools object

In [55]:
from toyaikit.tools import Tools

tools = Tools()
tools.add_tool(search_lessons)

# Cell 3 — Create OpenAI client

In [56]:
from openai import OpenAI
from toyaikit.llm import OpenAIClient

llm_client = OpenAIClient(
    model="gpt-5.4-mini",
    client=OpenAI()
)

# Cell 4 — Create assistant

In [58]:
from toyaikit.chat import ChatAssistant
from toyaikit.chat import IPythonChatInterface

assistant = ChatAssistant(
    tools=tools,
    developer_prompt="""
You're a course teaching assistant.

Answer the student's question using the search tool.

Make multiple searches with different keywords before answering.
""",
    chat_interface=IPythonChatInterface(),
    llm_client=llm_client
)

# Cell 5 — Reset counter

In [59]:
search_calls = 0

# Cell 6 — Run

In [60]:
assistant.run(
    "How does the agentic loop work, and how is it different from plain RAG?"
)

TypeError: ChatAssistant.run() takes 1 positional argument but 2 were given

In [61]:
import inspect

print(inspect.signature(assistant.run))

()


In [62]:
help(assistant.run)

Help on method run in module toyaikit.chat.chat:

run() method of toyaikit.chat.chat.ChatAssistant instance



In [63]:
import inspect

print(inspect.getsource(ChatAssistant.run))

    def run(self):
        self.runner.run()



In [64]:
print(inspect.signature(ChatAssistant.run))

(self)


In [65]:
assistant.run()

You: How does the agentic loop work, and how is it different from plain RAG?


-> Response received


-> Response received


You: bye


-> Response received


You: exit


-> Response received


You: quit


-> Response received


KeyboardInterrupt: Interrupted by user

In [66]:
print(search_calls)

3


# Complete